# The same architecture, one component swapped

From [jmurray10/agent_design_peas](https://github.com/jmurray10/agent_design_peas).

The argument of the series in one sentence: **classical agent architectures are
not obsolete, and an LLM upgrades one component inside them rather than replacing
the architecture.**

Every section below is a pair. `before.py` is the classical algorithm, standard
library only, no key, no network. `after.py` is the same architecture with exactly
one component swapped for a model call. Run them next to each other and the claim
is either visible or it is false.

**No API key needed.** With none configured, `after.py` replays what a real model
actually returned to that exact prompt, recorded in the repository with the model
name and the date. There is no third mode: a prompt with no recording raises rather
than inventing an answer. Add your own key in the optional cell to run them live.

## Setup


In [ ]:
!git clone --depth 1 https://github.com/jmurray10/agent_design_peas.git peas 2>/dev/null || echo 'already cloned'
%cd peas
!pip install -q pyyaml jsonschema anthropic

### Optional: run them live instead of replaying

Skip this cell to watch the replay. Put a key in Colab's Secrets panel (the key
icon in the left sidebar) as `ANTHROPIC_API_KEY` rather than pasting it into a cell.

In [ ]:
import os

try:
    from google.colab import userdata
    key = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    key = None

if key:
    os.environ['ANTHROPIC_API_KEY'] = key
    os.environ['LLM_PROVIDER'] = 'anthropic'
    print('Live. Every after.py below reaches a real model.')
else:
    print('No key found. after.py will replay recorded real responses.')

## 1. Simple reflex: the rule table becomes a model call

The classical agent is a lookup table from percept to action. It is fast, total,
and completely undone by a percept nobody wrote a rule for.

Watch the puddle. `before.py` has no rule matching it, the match fails, and the
agent returns `no_op` -- not because doing nothing is right, but because nothing
in the table had anything to say.

`after.py` sees the same square described as "puddle and dirt mixed together" and
answers `suck`, tagged `[chosen by the model]`. Same architecture, same loop, same
actuator list. The rule table is the only thing that moved.

That tag matters more than the action. Run it a few times and the model does not
always answer the same way -- `01-reflex-agents/simple/README.md` records `no_op`
three times in four and `suck` once, which is the recording shipped here. A rule
table is wrong in the same way every time; a model is wrong differently each time,
and that is the trade being made.

In [ ]:
!python 01-reflex-agents/simple/before.py

In [ ]:
!python 01-reflex-agents/simple/after.py

## 2. Model-based reflex: state survives, the parse gets a fallback

Now the agent carries state between percepts. The model updates that state, and a
deterministic schema check decides whether the update is allowed to land.

The fallback is the part worth reading. When the model's state update fails its own
schema, the agent does not crash and does not accept it: it merges what it can and
says so out loud.

In [ ]:
!python 01-reflex-agents/model-based/before.py

In [ ]:
!python 01-reflex-agents/model-based/after.py

## 3. Goal-based: A star does not go anywhere

The search is the architecture. The model's job is to turn a sentence into a start
state, and then A star runs exactly as it always did.

`after.py` prints its own comparison against `before.py`: same plan, same number of
nodes expanded, from prose instead of a tuple. If the model had replaced the search,
those numbers would not match.

In [ ]:
!python 02-goal-based/search/before.py

In [ ]:
!python 02-goal-based/search/after.py

### The solver is imported, not reimplemented

Easy to claim, so the repository proves it: object identity plus a digest of the
solver's source, checked at runtime. `after.py` is running the same function object
`before.py` is.

In [ ]:
!python 02-goal-based/csp/verify_identical.py

## 4. Utility-based: three ways to put a model near an MDP

Value iteration converges to an optimal policy and can prove it. `after.py` runs
three ways to involve a model and prints which guarantee each one keeps:

- **PATH 1, `LLMStateEstimator`** -- the model reads the world into a state and the
  MDP is solved classically. The guarantee survives, because the algorithm still
  runs over a model it was handed.
- **PATH 2, `LLMPolicyAgent`** -- the model is the policy. Nothing converges and
  nothing is optimal; it runs on state spaces value iteration cannot touch.
- **PATH 3, `LLMExplorationAgent`** -- the model explores and deterministic code
  keeps the books, for when the transition probabilities are unknown.

Only the first keeps the optimality guarantee. Which one is right depends on which
constraint you actually have, and that is the decision the section is about.

`03-utility-based/value-iteration/real_world.py` takes path one to a maintenance
schedule, where the model prices a breakdown out of a plant manager's paragraph and
the solver picks the policy.

In [ ]:
!python 03-utility-based/value-iteration/before.py

In [ ]:
!python 03-utility-based/value-iteration/after.py

## 5. Learning: four components, and the one that must not be a model

A learning agent has a performance element, a critic, a learning element, and a
problem generator. The model can serve three of them.

The critic is the interesting one. Where the reward is computable, compute it: a
deterministic critic is free, instant, reproducible, and cannot reorder outcomes.
The repository tested whether an LLM critic actually degrades the agent, twice, and
the answer was no -- which is why the claim it makes is narrower than the one it set
out to make. See `10-drift/critic-experiment/analysis.md`.

In [ ]:
!python 04-learning/q-learning/before.py

In [ ]:
!python 04-learning/q-learning/after.py

## 6. Multi-agent: alpha-beta prunes the same tree

The cell below runs `before.py`, which has no model in it at all. That is the
point of running it here: the node counts are the baseline everything else is
measured against, and they are deterministic -- the same on any machine, because
they count operations rather than seconds.

In `after.py` the model scores leaf positions and nothing else. The search still
prunes, and prunes identically, because pruning is a property of the algorithm
rather than of the evaluation function feeding it.

`05-multi-agent/adversarial/real_world.py` puts the same search on a price war,
where a leaf has no arithmetic value and the model supplies one per position.

In [ ]:
!python 05-multi-agent/adversarial/before.py

### The contract between agents is the architecture

The last run in this one is the payload: an extractor returns valid, plausible JSON,
and a deterministic schema check between the agents halts the pipeline before the
next agent is ever called. Nothing about the output looks wrong. It simply does not
satisfy the contract the next agent declared.

In [ ]:
!python 05-multi-agent/orchestration/after.py

## What stayed deterministic

In every pair above, the algorithm did not move. A star still expands the same
nodes. Alpha-beta still prunes the same tree. Value iteration still converges.
The CSP solver is the same function object, proved rather than asserted.

What the model replaced each time was a single component: the rule table, the state
update, the translation from prose to a start state, the reward function, the leaf
evaluation. That is the oscillation the series is about -- deterministic, model,
deterministic -- and the deterministic halves are what make the model's contribution
safe to accept.

### Where to go next

- `colab/agents_live.ipynb` -- nine of these architectures as config-driven agents,
  each behind an HTTP endpoint, all through one class with no agent-specific code.
- `colab/gpu_floor.ipynb` -- the parallelization floor measured on your own GPU.
- `10-drift/` -- what happens to all of this when the prompt moves three words.

If you ran this without a key, you watched a replay of real model output rather than
a simulation of one. That is not the same as running your own, and the repository
says so in its README rather than pretending otherwise.